### SetUp

In [2]:
# Download and unzip pigs-tracking.zip
!gdown https://drive.google.com/uc?id=1aa7_wxMHVVX1b1Djfgp2JoA8R5Xc19cI -O /content/processed_videos.zip
!unzip /content/processed_videos.zip -d /content

# Download and unzip pigs-yolov8-weights.zip
!gdown https://drive.google.com/uc?id=1YTbpI-m27oDJdGivKA1x01MA37Kv8_5- -O /content/pigs-yolov8-weights.zip
!unzip /content/pigs-yolov8-weights.zip -d /content

Downloading...
From (original): https://drive.google.com/uc?id=1aa7_wxMHVVX1b1Djfgp2JoA8R5Xc19cI
From (redirected): https://drive.google.com/uc?id=1aa7_wxMHVVX1b1Djfgp2JoA8R5Xc19cI&confirm=t&uuid=e32ffb3b-875d-4d0f-928f-8bcd482f500a
To: /content/processed_videos.zip
100% 250M/250M [00:01<00:00, 209MB/s]
Archive:  /content/processed_videos.zip
   creating: /content/content/processed_videos/
  inflating: /content/content/processed_videos/Test Vid 1_processed.mp4  
  inflating: /content/content/processed_videos/Test Vid 7_processed.mp4  
  inflating: /content/content/processed_videos/Test Vid 6_processed.mp4  
  inflating: /content/content/processed_videos/Test Vid 8_processed.mp4  
  inflating: /content/content/processed_videos/Test Vid 3_processed.mp4  
  inflating: /content/content/processed_videos/Test Vid 5_processed.mp4  
  inflating: /content/content/processed_videos/Test Vid 2_processed.mp4  
Downloading...
From (original): https://drive.google.com/uc?id=1YTbpI-m27oDJdGivKA1x01MA3

In [3]:
!pip install -qU ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.3 MB/s eta 0:00:00


### YOLO + SAM2 Video Segmentation Pipeline

In this notebook, we will build a pipeline that:

1. **Trims** a video to the first $N$ frames (default: $250$).
2. **Detects objects** in the first frame using YOLOv8.
3. **Segments** the detected objects across the trimmed video using SAM2.
4. **Saves** the final segmented video as output.

This approach ensures faster processing and prevents out-of-memory errors when working with long videos.


In [ ]:
import cv2
from ultralytics import YOLO
from ultralytics.models.sam import SAM2VideoPredictor

# -----------------------------
# Config
# -----------------------------
YOLO_MODEL_PATH = "/content/v9/weights/best.pt"
SAM_MODEL_PATH = "sam2.1_b.pt"
INPUT_DIR = "/content/content/processed_videos"
VIDEO_LIST = ["Test Vid 2_processed.mp4"]

SKIP_FRAMES = 50   # Skip first 50 frames
KEEP_FRAMES = 250  # Keep next 250 frames

# -----------------------------
# Load models
# -----------------------------
yolo_model = YOLO(YOLO_MODEL_PATH)
overrides = dict(conf=0.25, task='segment', mode='predict', imgsz=1024, model=SAM_MODEL_PATH)
predictor = SAM2VideoPredictor(overrides=overrides)

In [15]:
# -----------------------------
# Helper: Trim and resize video
# -----------------------------
def trim_and_resize_video(input_path, skip_frames=0, keep_frames=250, scale=0.5):
    """
    Trim video and reduce resolution by `scale`.
    - skip_frames: number of initial frames to skip
    - keep_frames: number of frames to keep after skipping
    - scale: fraction to resize width and height (0.5 = half size)
    """
    output_path = input_path.replace(".mp4", f"_trimmed_resized.mp4")

    cap = cv2.VideoCapture(input_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) * scale)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * scale)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0
    saved_count = 0

    while cap.isOpened() and saved_count < keep_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Skip initial frames
        if frame_count >= skip_frames:
            # Resize frame
            frame_resized = cv2.resize(frame, (width, height))
            out.write(frame_resized)
            saved_count += 1

        frame_count += 1

    cap.release()
    out.release()
    print(f"✅ Trimmed and resized video saved: {output_path}")
    return output_path

# -----------------------------
# Object detection + SAM on resized video
# -----------------------------
def detect_objects_resized(video_path, skip_frames=10, keep_frames=250, scale=0.5):
    """
    Trim, resize video, detect objects on first frame, segment with SAM2.
    """
    # ---- Step 1: Trim and resize ----
    processed_path = trim_and_resize_video(video_path, skip_frames, keep_frames, scale)

    # ---- Step 2: Run YOLO on first frame ----
    cap = cv2.VideoCapture(processed_path)
    ret, first_frame = cap.read()
    cap.release()

    if not ret:
        print("❌ Could not read first frame.")
        return

    results = yolo_model(first_frame, verbose=False)

    # ---- Step 3: Collect YOLO box centers ----
    points, labels = [], []
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        points.append([cx, cy])
        labels.append(1)  # foreground

    if not points:
        print("⚠️ No detections found in first frame.")
        return

    # ---- Step 4: Run SAM2 predictor ----
    sam_results = predictor(
        source=processed_path,
        points=points,
        labels=labels,
    )

    print("✅ Segmented resized video saved (check /content/runs/segment/predict{N})")

# -----------------------------
# Run pipeline on video list
# -----------------------------
for video_name in VIDEO_LIST:
    video_path = f"{INPUT_DIR}/{video_name}"
    detect_objects_resized(video_path, skip_frames=50, keep_frames=250, scale=0.5)

✅ Trimmed and resized video saved: /content/content/processed_videos/Test Vid 2_processed_trimmed_resized.mp4

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/250) /content/content/processed_videos/Test Vid 2_processed_trimmed_resized.mp4: 1024x1024 1 0, 1 1, 0.4ms
video 1/1 (frame 2/250) /content/content/processed_videos/Test Vid 2_processed_trimmed_resized.mp4: 1024x1024 1 0, 1 1, 475.9ms
video 1/1 (frame 3/250) /content/content/processed_videos/Test Vid 2_processed_trimmed_resiz

### RUN ALL
### Trim and Resize Video Function Parameters

The `trim_and_resize_video` function allows you to control **which frames to keep** and the **output resolution**.

### Parameters:

1. **`skip_frames`** (int, default=0)  
   - Number of frames to skip from the start of the video.  
   - Example: `skip_frames=50` → the first 50 frames are ignored.  
   - Skipped frames do **not appear** in the output video.

2. **`keep_frames`** (int, default=250)  
   - Number of frames to keep **after skipping**.  
   - Example: `keep_frames=250` → only 250 frames after skipping are saved.  
   - Determines the **length of the output video**.

3. **`scale`** (float, default=0.5)  
   - Factor to resize width and height of each frame.  
   - Example: `scale=0.5` → halves both width and height.  
   - Smaller values reduce resolution, speed up processing, and reduce memory usage.

### Notes:

- Increasing `skip_frames` skips more of the initial content.  
- Increasing `keep_frames` makes the output video longer.  
- Decreasing `scale` reduces resolution and file size, but may lose fine details.  
- Together, these parameters let you control **which part of the video is processed** and the **processing load** for object detection and segmentation.


In [ ]:
# VIDEO_LIST = [
#     "Test Vid 1_processed.mp4",
#     "Test Vid 6_processed.mp4",
#     "Test Vid 2_processed.mp4",
#     "Test Vid 7_processed.mp4",
#     "Test Vid 3_processed.mp4",
#     "Test Vid 8_processed.mp4",
#     "Test Vid 5_processed.mp4",
# ]


# for video_name in VIDEO_LIST:
#     video_path = f"{INPUT_DIR}/{video_name}"
#     detect_objects_resized(video_path, skip_frames=50, keep_frames=250, scale=0.5)